# Let's control the latest AI with programs! (Nano Banana / API Practice)


<table>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/AseiSugiyama/iput-bwai-handson/blob/main/handson_notebook_en.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/AseiSugiyama/iput-bwai-handson/blob/main/handson_notebook_en.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

Today, we will experience "operating the latest multimodal AI from a program" using Google Colab.
Let's unravel what's happening behind the AI we usually see in the browser by actually moving our hands.

## 1. Installation of necessary tools (SDK)

In [ ]:
!pip install --upgrade google-genai

### 2. Authentication for using Google Cloud

In today's hands-on, we will use Google Cloud.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
print("Preparation complete!")


## What is an API?

Today's goal is to understand how to freely call and use the functions of convenient services such as Vertex AI from your programs, not from a web browser.

An API (Application Programming Interface) is essential for that.

### Definition of API

An API is a set of rules (promises) for making the functions of one software available from another external program.

### Role of API

1. Window to other software: Provides convenient functions of a service or software to other software.
2. Request and Response: Strictly defines in what format to request and in what format the result will be returned.
3. Hiding Implementation: Results can be obtained by asking as decided without knowing the internal implementation.

### Web API Mechanism

There are various types of APIs, but today we will use Web API (REST API), which uses functions over the Internet.

The biggest feature of Web API is that it uses the same HTTP mechanism used when you normally view websites.

Therefore, as the first step to controlling the latest AI, let's start by unraveling the true identity of the Web you usually use.

## The true identity of a Web page is a collection of Text

The Web pages we usually see in browsers (like Chrome) are made of text data consisting of symbols.

The browser just reads that text and colors it or places images so it's easy for humans to see.

First, let's directly obtain that raw text data through the Internet.

### Try obtaining the content of example.com with a command without using a browser

[example.com](http://example.com) is a special site for use as a sample in documentation, etc. First, open this site in your browser.

Next, let's display the original text in the browser. In Chrome, you can check it by "View page source" or starting developer tools.

Now, let's execute the same thing with a command. Executing the following command will access the server and obtain the source of the Web page.

In [ ]:
!curl -v http://example.com

The execution result will be as follows.

```
curl -v http://example.com
* Host example.com:80 was resolved.
* IPv6: 2606:4700:10::ac42:93f3, 2606:4700:10::6814:179a
* IPv4: 104.20.23.154, 172.66.147.243
*   Trying [2606:4700:10::ac42:93f3]:80...
* Connected to example.com (2606:4700:10::ac42:93f3) port 80
> GET / HTTP/1.1
> Host: example.com
> User-Agent: curl/8.7.1
> Accept: */*
> 
* Request completely sent off
< HTTP/1.1 200 OK
< Date: Tue, 05 May 2026 05:32:30 GMT
< Content-Type: text/html
< Transfer-Encoding: chunked
< Connection: keep-alive
< Server: cloudflare
< Last-Modified: Fri, 01 May 2026 01:24:29 GMT
< Allow: GET, HEAD
< Accept-Ranges: bytes
< Age: 4015
< cf-cache-status: HIT
< CF-RAY: 9f6d5d727e18f6fe-NRT
< 
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>
* Connection #0 to host example.com left intact
```

[cURL](https://curl.se/) is a tool for interacting with data over the Internet from the command line without using a browser.

Let's look at the contents of the execution result in detail. HTTP communication is roughly divided into two parts: "Request" and "Response".

### 1. Request
> Lines starting with > are what was sent from your PC to the server.
- GET / HTTP/1.1: A request to GET (give me) the / (top page).
- Host: example.com: The name of the server you are asking.

### 2. Response
< Lines starting with < are what came back from the server to your PC.
- HTTP/1.1 200 OK: 200 is the code (status code) for success.
- Content-Type: text/html: Indicates that the returned data is text in HTML format.

### 3. Body
The long text starting with <!doctype html>... displayed at the end is the content of the communication itself. When a browser reads this, it becomes the Web site screen we know well.

In this way, we found that Web site information can be obtained just by conversation in text (HTTP communication) without using a browser. Web API uses exactly the same mechanism.


## Common rules for talking to APIs

In order to use external functions as parts through Web API, you need to follow common rules decided worldwide. That is the concept of REST API.

In REST API, communication is mainly performed by combining the following two elements.

### 1. HTTP Methods (Verbs)
Symbols that represent what you want to do to the server.
- GET (Retrieve): Used when asking for information. (e.g., viewing a web page, getting cat trivia)
- POST (Create/Process): Used when asking to process by sending data from here. (e.g., asking an AI to create an image by sending a prompt)

### 2. JSON (Noun/Data Format)
A common language for computers to interact with data efficiently.
The HTML we saw earlier contained layout information for humans, but JSON (JavaScript Object Notation) is a simple text format containing only pure data.

Next, let's experience the difference from HTML by hitting a real API that returns only data.

### Practice 1: Let's retrieve information from a public API (GET)

Here we will use a public API called catfact.ninja. This is a simple service that returns one random "cat trivia" every time it is executed.

Since we are asking to "give me" information, we use GET as the HTTP method.

In [ ]:
!curl https://catfact.ninja/fact


When executed, text like the following is returned (the content changes every time it is executed).

```json
{"fact":"Statistics indicate that animal lovers in recent years have shown a preference for cats over dogs!","length":98}
```

This is data in JSON format. JSON has the following format.

```json
{
    "key1":"value1",
    "key2":"value2",
    :
}
```

Keys ("key1", "key2") and values ("value1", "value2") can be set freely. Also, various data such as numbers can be specified as values besides strings. This time it is as follows.

- fact: Body of the trivia
- length: Length of that text

Unlike HTML, you can see that there are no unnecessary decorations, and "item name" and "content" are lined up in sets so that computers can easily process them.

JSON specifications can be confirmed at the following site.

[https://www.json.org/](https://www.json.org/)

### Practice 2: Let's send data from here (POST)

Next, experience a POST request to send data from here to the server. This is the same mechanism as the communication when requesting "Create an image with this prompt!" from an AI.

Let's send JSON data using the test service `httpbin.org`.


In [ ]:
# -X POST: Designation to use the POST method
# -H ...: Declaration (header) that "I'm going to send JSON now"
# -d ...: Content to send (data)
!curl -X POST -H "Content-Type: application/json" -d '{"message": "hello IPUT!"}' https://httpbin.org/post

The execution result will be as follows.

```json
{
  "args": {}, 
  "data": "{\"message\": \"hello IPUT!\"}", 
  "files": {}, 
  "form": {}, 
  "headers": {
    "Accept": "*/*", 
    "Content-Length": "26", 
    "Content-Type": "application/json", 
    "Host": "httpbin.org", 
    "User-Agent": "curl/8.7.1", 
    "X-Amzn-Trace-Id": "Root=1-69f987b6-63ddd29d7aa954bb692afa6b"
  }, 
  "json": {
    "message": "hello IPUT!"
  }, 
  "origin": "123.225.37.37", 
  "url": "https://httpbin.org/post"
}
```

In the execution result, you should find a part like the following.

```json
"json": {
    "message": "hello IPUT!"
}
```

This is the server telling you "I have correctly received the JSON data (hello IPUT!) you sent".

In this way, by using an API, it becomes possible not only to "retrieve information" but also to "send data from here and have it processed".

### Python Dictionaries and JSON

In the previous practice, we specified data like `{"message": "hello IPUT!"}` when sending a POST request.

This way of writing is the same as the **dictionary type (dict)** in Python.

#### Common language of data

- **JSON**: Common rule of "text format" for interacting with data on the Internet.
- **Dictionary type**: A type of data in Python. Although it can be described in text in a program, it is strictly different from a string in Python.

Behind the program using the API, the following conversions are always taking place.

1. **Serialization (Conversion)**: Convert Python data to JSON (just a string) and send it to the Internet.
2. **Deserialization (Restoration)**: Convert JSON (just a string) arriving from the Internet back to Python data and use it in the program.

Next, let's actually write Python code and perform this "conversion" and "communication" automatically.

### Practice 3: Let's hit the API with Python code (GET)

Finally, access the Internet from a program (Python). Here, we use a library called `requests`, which is often used for Web communication.

Let's write the retrieval of "cat trivia" that we did with `curl` earlier in Python.

In [ ]:
import requests

# 1. Send request (GET)
response = requests.get('https://catfact.ninja/fact')

# 2. Convert (parse) the arrived JSON (text) into Python data (dictionary)
data = response.json()

# 3. Check the content
print(f"Raw data: {response.text}")
print(f"Converted data: {data}")
print(f"Extracting only trivia: {data['fact']}")

#### Code Points

- `requests.get()`: Communicates with the specified URL to "ask for information".
- `response.json()`: Automatically analyzes (parses) the arrived JSON string and turns it into a Python dictionary.

As a result, data on the Internet can now be handled in the same way as the Python lists and dictionaries you know.

### Practice 4: Let's send data with Python code (POST)

Next, let's send data to the server using Python code.
I'll write the same content as what I sent with the `curl` command earlier using the `requests` library.

In [ ]:
import requests

# Data to send (dictionary type)
payload = {"message": "hello from Python!"}

# Send request (POST)
# By passing a dictionary to the json= argument, it will automatically be converted to a JSON string and sent
response = requests.post('https://httpbin.org/post', json=payload)

# Check results
data = response.json()
print(f"The sent data was recognized by the server: {data}")
print(f"The sent data is sent back as follows: {data['json']}")

### Code Points

- `json=payload`: With just this, it serializes the Python dictionary into JSON and sends it.
- `data['json']`: Checking the part where the server (httpbin.org) organizes and returns the "received JSON".

With this, we were able to send data from "the code we wrote" to an "external function" via the Internet and receive the result. Basically the same thing happens when sending a prompt to an AI.

## Let's look at the official reference

It is becoming clear that the true identity of an API is "conversation in text".
Now, let's look at the API specifications of Google Cloud's latest AI "Gemini", which is the main character of today.

### Gemini API (generateContent) Specifications

The formal rules for asking an AI to "Generate text or images!" are described in the following official documentation.

[Vertex AI API Reference (generateContent)](https://cloud.google.com/vertex-ai/generative-ai/docs/reference/rest/v1/projects.locations.publishers.models/generateContent)

### Let's look at the document

Open the URL and try to find the item called Request body. It describes a very complex and deeply hierarchical JSON structure necessary to send a single instruction to the AI.

Always making a dictionary by hand and sending it using the requests library will lead to the following problems.

- The hierarchy is too deep, and you might make a mistake in where to write what.
- It's a hassle to manage project IDs and authentication tokens every time.

"I want to use APIs more easily and smartly!"
Next, I will introduce the SDK (Software Development Kit), which is a tool to grant such developer wishes.

## Let's use the SDK

As we saw earlier, dealing with raw Web APIs directly is very time-consuming, such as constructing URLs and creating complex JSON.

That's where the **SDK (Software Development Kit)** comes in.

### What is an SDK?

An SDK is a "toolbox (library)" for easily using a specific Web API from a specific programming language.

In this workshop, it functions as a **"dedicated remote control (wrapper)"** for freely operating Web APIs from Python.

### Benefits of using SDK

1. **Automation of complex procedures**: Constructing URLs, adding authentication headers, converting (parsing) to JSON, etc., are all done automatically behind the scenes.
2. **Intuitive code**: Complex communication that took dozens of lines in `requests` can be executed by just calling one Python function.
3. **Prevention of mistakes**: Since there is no need to hand-write deeply hierarchical JSON, input mistakes and structural errors can be dramatically reduced.

From now on, let's introduce the Google official SDK and master the "hopelessly complex API" we saw earlier.

### Let's connect to AI using SDK (Create Client)

Finally, prepare to connect to Google's latest AI. Here, we use genai.Client to create your own client.

#### Why create a Client?

To use an API, you need to tell the server "which project to use (Project ID)", "where to communicate (Location)", "Authentication password", etc., every time. It's hard to write these every time without omission.

Therefore, we create "your dedicated AI communication representative (client)" that remembers all these settings in a determined way.

After this, by just giving instructions to this representative such as "Create text" or "Create image", all the troublesome communication procedures will be automatically done behind the scenes.

In [ ]:
from google import genai
from google.colab import auth

# Execute authentication (a popup will appear here if not already done)
auth.authenticate_user()

# Enter your Project ID
PROJECT_ID = "YOUR_PROJECT_ID" # @param {type:"string"}

# Initialize SDK client (representative)
client = genai.Client(
    vertexai=True, 
    project=PROJECT_ID, 
    location="global",
)

print(f"Preparation of AI connection representative (client) is complete in project {PROJECT_ID}!")

## Image Generation using Gemini (nanobanana)

Based on the content so far, let's actually generate images using Gemini (Nano Banana).

Master how to perform image generation, which you usually do from an app, from a program.

### Today's Practice Flow

The method for performing image generation is also the same as the method for using Web APIs that we have seen so far. Specifically, we will do the following.

1. Tell the AI "what kind of image you want to create" (Request).
2. The AI generates an image while explaining in words (Response).
3. Extract and display image data from the arrived data.

Now, let's incorporate the world's most advanced AI into your program!

### Practice: Let's have Gemini generate an image

Now, actually send a prompt to generate an image.
In this code, "Who and how to ask" and "What materials to pass" are roughly defined.

#### Who and how to ask (Specification of communication destination)

- `client.models`:
  A specialized department for handling AI models held by the dedicated representative (client) you created earlier.
- `generate_content(...)`:
  A function that executes an order (POST request) to "Create content!" to that specialized department. Pass the following "materials (arguments)" inside these parentheses `()`.

#### What materials to pass (Arguments of generate_content)

- Argument `model=`: Specifies which AI model to use. Here, the image generation model put in `MODEL_ID` is specified.
- Argument `contents=`: The body of the message to send to the AI. Here, the text "A cyberpunk cat glowing blue" put in the `prompt` variable is passed.
- Argument `config=`: A "detailed specification sheet" to send to the API. This time, we use `GenerateContentConfig` to give special permission to "output images (`Modality.IMAGE`) along with text".

Creating an object that summarizes settings and passing it as an argument might feel strange until you get used to it. This method is used when there are too many setting items, they are complex with a hierarchical structure, or there are many default values. If you want to make it behave slightly differently from the default, it would be good to look up what kind of settings can be made using such an object.

Now, let's execute the code and send a request to the AI.

In [ ]:
from google.genai.types import GenerateContentConfig, Modality

# Name of the model to use this time
MODEL_ID = "gemini-3.1-flash-image-preview-001"

# Write a description of the image you want to create
prompt = "A cyberpunk cat glowing blue"

print("Request sent to AI. Drawing while thinking... (takes a few dozen seconds)")

# Send request to AI through the dedicated representative (client)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=GenerateContentConfig(
        # Setting to allow AI to return both "text" and "image"
        response_modalities=[Modality.TEXT, Modality.IMAGE],
    ),
)

print("Generation complete and response has arrived! Let's look inside next.")

### "Dissection" of the Response

Generation is complete!
Now, let's peek inside the data (`response`) returned from the AI.

In [ ]:
# Try displaying the entire returned data
print(response)


It looks like a very complex structure, but in fact, the SDK has already organized the output from the server as a "Python object".

When we used `requests` in Step 2.5, we had to write the process (parsing) of `response.json()` ourselves. However, if you use the SDK, that process is already finished.

To extract necessary data, you can access it intuitively by just connecting with a dot (`.`), not by searching like a dictionary.

In [ ]:
print(f"Total token count consumed: {response.usage_metadata.total_token_count}")

### Let's save the generated image as a file

The response contains not only text but also "image data".
However, the data that arrived from the AI (`inline_data.data`) is raw data for computers, that is, "just a list of 0 and 1 numbers (binary data)".
As it is, it cannot be displayed as a "picture" on the screen or saved as a file.

So, we use two special tools.

1. BytesIO: A tool for handling a list of numbers in a program "as if an image file (.png, etc.) were there". It's like creating a "virtual file" on memory.
2. PIL (Python Imaging Library): A tool that reads that virtual file, converts it into an "image" that we can see with our eyes, writes it as a file, or displays it on the screen.


Once the data arriving from the AI is converted to an image with `PIL`, let's **"save it as a file"**.
By leaving it as a file instead of just displaying it on the screen, you can download it later or use it in other apps.

AI sometimes generates multiple images at once.
At that time, if there is only one file name, it will be overwritten, so let's save it by numbering it in order using the Python function called enumerate.

In [ ]:
from PIL import Image
from io import BytesIO

print("--- Saving generated images ---")

# By using enumerate, you can get "what number (i)" together with the content (part)
for i, part in enumerate(response.candidates[0].content.parts):
    if part.inline_data:
        # 1. Handle the list of numbers arriving from AI as a virtual file (BytesIO)
        # 2. Read it as an image (Image.open)
        img = Image.open(BytesIO(part.inline_data.data))

        # Alternatively, another convenient way
        # img = part.inline_data.as_image()

        # 3. Save it by numbering the file name (utilizing f-string)
        # Example: output_image_0.png, output_image_1.png ...
        filename = f"output_image_{i}.png"
        img.save(filename)
        
        print(f"Saved image as {filename}!")

        # 4. Also display on screen to check
        display(img)

With this, a series of flows from giving instructions to the AI from the "code you wrote" to receiving results and "getting them as files" has been completed!

## Good job!

Today, we ran through everything from the mechanism called "Web API" behind the browser to how to use the "SDK" to freely manipulate the latest AI models with Python.

### What you can do now

- Talk to the Internet from a program without going through a browser.
- Handle complex JSON data as a Python dictionary.
- Give instructions to the latest AI using SDK (dedicated remote control), generate images, and save them to files.

### Why use AI from a "program"?

Recall the "Capcom case" introduced in the slides at the beginning.
If you were inputting tens of thousands of object proposals one by one in a browser to create images, it would take months.
However, as you learned today, if you use a **program (API)**, you can finish the task of reading tens of thousands of data in a list, automatically giving instructions to the AI, and automatically saving to files in a few hours.

This is the greatest weapon you gain by learning programming.
"I wanted this kind of image automatically" "Can I leave this task to AI?"
Keep such curiosity in mind, and please use the latest AI to the fullest and create interesting things!

I'm looking forward to meeting your wonderful works!